In [1]:
import numpy as np
import gymnasium as gym
from gymnasium.utils.env_checker import check_env

In [2]:
def simulate_tracks(
    n_planes,
    spatial_resolution,
    n_tracks=1000,
    plane_size=100.0,
    start_area_fraction=0.10,
    z_min=0.0,
    z_max=600.0,
    theta_max_deg=10.0,
    beta_min=0.5,
    beta_max=0.75,
    time_resolution=2.0,
    tof_margin=50.0,
    default_value=-999999.0,
    seed=42
):
    """
    Simula tracce Monte Carlo con:
    - n_planes piani di tracciamento sensibili a x,y
    - 2 piani temporali, uno prima e uno dopo il tracciatore
    - velocità casuale tra beta_min*c e beta_max*c
    - controllo dell'accettanza geometrica dei piani

    Le hit fuori accettanza vengono riempite con default_value.

    Returns
    -------
    x_meas : np.ndarray
        Coordinate x misurate, shape = (n_tracks, n_planes).
        Se fuori accettanza: default_value.

    y_meas : np.ndarray
        Coordinate y misurate, shape = (n_tracks, n_planes).
        Se fuori accettanza: default_value.

    z_planes : np.ndarray
        Coordinate z dei piani di tracciamento, shape = (n_planes,)

    theta : np.ndarray
        Angolo polare generato rispetto all'asse z, in radianti.

    theta_deg : np.ndarray
        Angolo polare generato rispetto all'asse z, in gradi.

    beta : np.ndarray
        Velocità generata in unità di c.

    delta_t_meas : np.ndarray
        Differenza di tempo misurata tra i due piani temporali, in ns.

    hit_valid : np.ndarray
        Maschera booleana, shape = (n_tracks, n_planes).
        True se la hit è dentro l'accettanza del piano.
    """

    rng = np.random.default_rng(seed)

    # Velocità della luce in mm/ns
    c_mm_ns = 299.792458

    # Posizione dei piani di tracciamento
    z_planes = np.linspace(z_min, z_max, n_planes)

    # Posizione dei due piani temporali
    z_time_1 = z_min - tof_margin
    z_time_2 = z_max + tof_margin

    delta_z_tof = z_time_2 - z_time_1

    # Regione iniziale pari a start_area_fraction dell'area del piano
    start_size = plane_size * np.sqrt(start_area_fraction)
    start_half_size = start_size / 2

    x0 = rng.uniform(-start_half_size, start_half_size, n_tracks)
    y0 = rng.uniform(-start_half_size, start_half_size, n_tracks)

    # Angoli casuali entro un cono attorno all'asse z
    theta_max = np.deg2rad(theta_max_deg)

    phi = rng.uniform(0, 2 * np.pi, n_tracks)

    # Distribuzione uniforme in angolo solido
    cos_theta = rng.uniform(np.cos(theta_max), 1, n_tracks)
    theta = np.arccos(cos_theta)
    theta_deg = np.rad2deg(theta)

    tan_theta = np.tan(theta)

    tx = tan_theta * np.cos(phi)   # dx/dz
    ty = tan_theta * np.sin(phi)   # dy/dz

    # Velocità casuale
    beta = rng.uniform(beta_min, beta_max, n_tracks)

    # Hit vere sui piani di tracciamento
    x_true = x0[:, None] + tx[:, None] * z_planes[None, :]
    y_true = y0[:, None] + ty[:, None] * z_planes[None, :]

    # Maschera di accettanza geometrica
    half_plane_size = plane_size / 2

    hit_valid = (
        (np.abs(x_true) <= half_plane_size) &
        (np.abs(y_true) <= half_plane_size)
    )

    # Hit misurate con risoluzione spaziale gaussiana
    x_meas = x_true + rng.normal(
        0,
        spatial_resolution,
        size=x_true.shape
    )

    y_meas = y_true + rng.normal(
        0,
        spatial_resolution,
        size=y_true.shape
    )

    # Se la traccia è fuori accettanza, il piano non misura nessuna hit
    x_meas[~hit_valid] = default_value
    y_meas[~hit_valid] = default_value

    # Tempo di volo vero tra i due piani temporali
    path_length = delta_z_tof / np.cos(theta)
    delta_t_true = path_length / (beta * c_mm_ns)

    # Misura del tempo sui due piani temporali
    t1_meas = rng.normal(0, time_resolution, n_tracks)
    t2_meas = delta_t_true + rng.normal(0, time_resolution, n_tracks)

    delta_t_meas = t2_meas - t1_meas

    return x_meas, y_meas, z_planes, theta, theta_deg, beta, delta_t_meas

In [3]:
import numpy as np
from gymnasium import spaces

class DetectorOptimizerEnv(gym.Env):
    def __init__(self):
        super().__init__()

        # --- SPAZIO DELLE AZIONI ---
        # [n_piani, risoluzione_spaziale, risoluzione_temporale, lunghezza_totale_z]
        # n_piani: da 3 a 20
        # risoluzione_spaziale: da 0.01 a 1.0 mm
        # risoluzione_temporale: da 0.1 a 5.0 ns
        # lunghezza_totale (z_max): da 100 a 1000 mm
        self.action_space = spaces.Box(
            low=np.array([2.0, 0.001, 0.01, 10.0]),
            high=np.array([20.0, 1.0, 5.0, 1000.0]),
            dtype=np.float32
        )

        # L'osservazione è lo stato del design attuale
        self.observation_space = self.action_space

        # Stato iniziale casuale
        self.state = self.action_space.sample()

    def step(self, action):
        # 1. Parsing delle azioni
        n_planes = int(np.round(action[0]))
        spatial_res = float(action[1])
        time_res = float(action[2])
        z_max_val = float(action[3])

        # 2. Esecuzione Simulazione Monte Carlo
        # (Usiamo la tua funzione simulate_tracks adattata)
        results = self._run_simulation(n_planes, spatial_res, time_res, z_max_val)
        
        # 3. Calcolo del Reward
        # Vogliamo minimizzare l'errore su beta (massimizzare -MSE)
        mse_beta = np.mean((results['beta_meas'] - results['beta_true'])**2)
        
        # Introduciamo un "Penalty Cost" per evitare design infinitamente costosi
        # Più piani = più costo | Risoluzione piccola = più costo
        cost = (n_planes * 1) + (spatial_res ** 1.5) + (time_res * 1)
        
        # Reward: Precisione meno Costo
        # Usiamo il logaritmo per l'errore per gestire diversi ordini di grandezza
        reward = -np.log10(mse_beta + 1e-9) - cost
        
        self.state = action
        return self.state, float(reward), False, False, {"mse_beta": mse_beta, "n_planes": n_planes}

    def _run_simulation(self, n_p, s_res, t_res, z_max):
        # Chiamata alla funzione originale fornita da te
        # Assumiamo n_tracks=200 per avere una statistica decente nel reward
        x_m, y_m, z_p, th, th_d, beta_true, dt_m = simulate_tracks(
            n_planes=n_p,
            spatial_resolution=s_res,
            time_resolution=t_res,
            z_max=z_max,
            n_tracks=200
        )
        
        # Ricostruzione brutale di beta per valutare il design
        c_mm_ns = 299.792458
        z_time_1 = 0.0 - 50.0 # tof_margin fisso
        z_time_2 = z_max + 50.0
        path_length = (z_time_2 - z_time_1) / np.cos(th)
        beta_meas = path_length / (dt_m * c_mm_ns)
        
        return {'beta_true': beta_true, 'beta_meas': beta_meas}

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.state = self.action_space.sample()
        return self.state, {}

In [4]:
import torch
from torchrl.envs.libs.gym import GymWrapper
from torchrl.envs import TransformedEnv, StepCounter

# 1. Creazione dell'ambiente
base_env = DetectorOptimizerEnv()
env = GymWrapper(base_env)

# 2. Aggiunta di trasformazioni
# Poiché l'azione ha range diversi (3-20 vs 0.01-1), 
# è utile normalizzarla tra -1 e 1 per la rete neurale.
#from torchrl.envs.transforms import DoubleToFloat, ActionScaling

env = TransformedEnv(env)
env.append_transform(StepCounter())

# Esempio di interazione:
td = env.reset()
# Supponiamo che la policy generi un'azione
action = torch.tensor([0.5, 0.8, 0.2, 0.1]) # Azione normalizzata

td["action"] = action
td = env.step(td)

print(f"Nuovo Design Proposto: {td['next', 'observation']}")
print(f"Reward (Qualità Design): {td['next', 'reward'].item()}")

/home/stelfano/anaconda3/envs/RL/lib/python3.13/site-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/home/stelfano/anaconda3/envs/RL/lib/python3.13/site-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


Nuovo Design Proposto: tensor([0.5000, 0.8000, 0.2000, 0.1000])
Reward (Qualità Design): -1.3757555484771729


In [5]:
rollout = env.rollout(1)
print("action:", rollout['action'])
print("reward:", rollout['next', 'reward'])
print("observation:", rollout['next', 'observation'])
print("initial observation:", rollout['observation'])

action: tensor([[1.7628e+01, 1.7426e-01, 1.3362e+00, 4.4930e+02]])
reward: tensor([[-20.3149]])
observation: tensor([[1.7628e+01, 1.7426e-01, 1.3362e+00, 4.4930e+02]])
initial observation: tensor([[ 8.6104,  0.6306,  0.4190, 22.2232]])


In [ ]:
from torch import nn
from torchrl.modules import MLP, Actor, AdditiveGaussianModule
from tensordict.nn import TensorDictSequential as Seq

# 1. Definiamo la rete per la policy (Actor)
# L'output deve corrispondere alla dimensione delle azioni continue
policy_mlp = MLP(
    out_features=env.action_spec.shape[-1], 
    num_cells=[128, 256, 1024, 128],
    activation_class = nn.Tanh,
    activate_last_layer=False
)

# Trasformiamo la MLP in un Actor che legge "observation" e scrive "action"
policy_module = Actor(
    module=policy_mlp,
    spec=env.action_spec,
    in_keys=["observation"],
    out_keys=["action"]
)

exploration_module = AdditiveGaussianModule(
    spec=env.action_spec,
    sigma_init=1.0,          
    sigma_end=1.0,           # Deviazione standard finale (poco rumore)
)

policy_explore = Seq(policy_module, exploration_module)

td = env.reset()
td = policy_explore(td)
print("Azione proposta dalla policy con esplorazione:", td['action'])

Azione proposta dalla policy con esplorazione: tensor([2.0000e+00, 1.0000e-03, 1.0000e-02, 1.0000e+01],
       grad_fn=<ClampBackward0>)


In [7]:
from torchrl.collectors import Collector
from torchrl.data import LazyTensorStorage, ReplayBuffer

init_rand_steps = 100
frames_per_batch = 100
optim_steps = 10
collector = Collector(
    env,
    policy_explore,
    frames_per_batch=frames_per_batch,
    total_frames=-1,
    init_random_frames=init_rand_steps,
)
rb = ReplayBuffer(storage=LazyTensorStorage(100_000))

from torch.optim import Adam

In [ ]:
from torchrl.modules import ValueOperator
from torch.optim import Adam

obs_dim = env.observation_spec["observation"].shape[-1]
act_dim = env.action_spec.shape[-1]
print(obs_dim, act_dim)

critic_mlp = MLP(
    in_features=obs_dim + act_dim, 
    out_features=1, 
    num_cells=[128, 256, 1024, 128],
    activation_class=nn.Tanh,
    activate_last_layer=False
)

critic_module = ValueOperator(
    module=critic_mlp,
    in_keys=["observation", "action"],
)

4 4


In [ ]:
from torchrl.objectives import DDPGLoss, SoftUpdate

loss = DDPGLoss(
    actor_network=policy_module,   
    value_network=critic_module,   
    delay_value=True               
)

loss.make_value_estimator(gamma=0.99)

optim = Adam(loss.parameters(), lr=0.02)

updater = SoftUpdate(loss, eps=0.99)

In [10]:
rollout = env.rollout(1, policy_explore)
print(rollout['action'])

tensor([[ 2.0000,  1.0000,  2.0196, 10.0000]], grad_fn=<StackBackward0>)


In [11]:
import time

total_count = 0
total_episodes = 0
t0 = time.time()

for i, data in enumerate(collector):
    # 1. Scrittura dei dati nel replay buffer
    rb.extend(data)
    rewards = data["next", "reward"]
    avg_reward = rewards.mean().item()
    
    print(f"Batch {i}: Reward Medio = {avg_reward:.4f}")
    # Calcoliamo la lunghezza massima raggiunta in questo episodio
    max_length = rb[:]["next", "step_count"].max()
    
    if len(rb) > init_rand_steps:
        for _ in range(optim_steps):
            # 2. Campionamento
            sample = rb.sample(128)
            
            # 3. Calcolo della Loss
            loss_vals = loss(sample)
            
            # NOTA: DDPGLoss restituisce 'loss_actor' e 'loss_value'.
            # Dobbiamo sommarle per fare un unico backward se l'optimizer è condiviso.
            total_loss = loss_vals["loss_actor"] + loss_vals["loss_value"]
            
            total_loss.backward()
            optim.step()
            optim.zero_grad()
            
            # 4. Aggiornamento fattore di esplorazione (Annealing dello sigma)
            # Usiamo i passi reali raccolti
            exploration_module.step(data.numel())
            
            # 5. Update dei parametri target (Soft Update)
            updater.step()
            
            if i % 10 == 0:
                print(f"Passi totali: {total_count}, Loss Actor: {loss_vals['loss_actor']:.4f}, Loss Critic: {loss_vals['loss_value']:.4f}")
        
            total_count += data.numel()
        total_episodes += data["next", "done"].sum()

    # Condizione di uscita basata sulla lunghezza del task
t1 = time.time()

2026-05-05 16:52:56,839 [torchrl][INFO]    Initialized LazyTensorStorage with torch.Size([100000]) shape [END]
Batch 0: Reward Medio = -15.0089
Batch 1: Reward Medio = -1.3488
Batch 2: Reward Medio = -6.6277
Batch 3: Reward Medio = -0.0425
Batch 4: Reward Medio = 1.2554
Batch 5: Reward Medio = 1.0631
Batch 6: Reward Medio = 1.0128
Batch 7: Reward Medio = 0.8135
Batch 8: Reward Medio = 0.9320
Batch 9: Reward Medio = 0.7832
Batch 10: Reward Medio = 0.7431
Passi totali: 9000, Loss Actor: 0.0652, Loss Critic: 46.1456
Passi totali: 9100, Loss Actor: -0.1082, Loss Critic: 43.8719
Passi totali: 9200, Loss Actor: -0.2385, Loss Critic: 33.4818
Passi totali: 9300, Loss Actor: -0.1128, Loss Critic: 49.6384
Passi totali: 9400, Loss Actor: 0.1102, Loss Critic: 47.7804
Passi totali: 9500, Loss Actor: 0.4138, Loss Critic: 27.7070
Passi totali: 9600, Loss Actor: 0.6602, Loss Critic: 71.6240
Passi totali: 9700, Loss Actor: 0.8614, Loss Critic: 55.0998
Passi totali: 9800, Loss Actor: 0.8428, Loss Critic

KeyboardInterrupt: 

In [ ]:
env.rollout(10)['action']

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

In [ ]:
#Disegna geometria


def draw_detector(
    z_planes,
    plane_size=100.0,
    tof_margin=50.0,
    x_hits=None,
    y_hits=None,
    n_tracks_show=20
):
    """
    Disegna il detector in 3D usando la coordinata fisica z
    come asse orizzontale del grafico.

    Mapping del plot:
        asse X grafico = z fisico
        asse Y grafico = x fisico
        asse Z grafico = y fisico
    """

    fig = plt.figure(figsize=(12, 6))
    ax = fig.add_subplot(111, projection="3d")

    half_size = plane_size / 2

    z_tracker_min = z_planes[0]
    z_tracker_max = z_planes[-1]

    z_time_1 = z_tracker_min - tof_margin
    z_time_2 = z_tracker_max + tof_margin

    def add_plane(z_phys, label, facecolor, alpha=0.25):
        """
        Piano fisico perpendicolare a z.
        Nel plot viene disegnato a X = z_phys,
        esteso lungo Y = x e Z = y.
        """

        vertices = [
            [
                (z_phys, -half_size, -half_size),
                (z_phys,  half_size, -half_size),
                (z_phys,  half_size,  half_size),
                (z_phys, -half_size,  half_size),
            ]
        ]

        plane = Poly3DCollection(
            vertices,
            facecolors=facecolor,
            edgecolors="black",
            linewidths=1.0,
            alpha=alpha
        )

        ax.add_collection3d(plane)

        ax.text(
            z_phys,
            half_size * 1.08,
            half_size * 1.08,
            label,
            fontsize=9
        )

    # Primo piano temporale
    add_plane(
        z_time_1,
        "TOF 1\nsolo tempo",
        facecolor="tab:orange",
        alpha=0.25
    )

    # Piani di tracciamento
    for i, z in enumerate(z_planes):
        add_plane(
            z,
            f"Tracker {i}",
            facecolor="tab:blue",
            alpha=0.18
        )

    # Secondo piano temporale
    add_plane(
        z_time_2,
        "TOF 2\nsolo tempo",
        facecolor="tab:orange",
        alpha=0.25
    )

    # Disegna alcune tracce
    if x_hits is not None and y_hits is not None:
        n_tracks = x_hits.shape[0]
        n_show = min(n_tracks_show, n_tracks)

        for i in range(n_show):
            ax.plot(
                z_planes,      # asse X grafico = z fisico
                x_hits[i],     # asse Y grafico = x fisico
                y_hits[i],     # asse Z grafico = y fisico
                marker="o",
                markersize=3,
                linewidth=1,
                alpha=0.8
            )

    # Limiti degli assi
    z_min_plot = z_time_1 - 0.1 * abs(z_time_2 - z_time_1)
    z_max_plot = z_time_2 + 0.1 * abs(z_time_2 - z_time_1)

    ax.set_xlim(z_min_plot, z_max_plot)
    ax.set_ylim(-half_size, half_size)
    ax.set_zlim(-half_size, half_size)

    ax.set_xlabel("z fisico [mm]")
    ax.set_ylabel("x fisico [mm]")
    ax.set_zlabel("y fisico [mm]")

    ax.set_title("Schema 3D del detector con asse z orizzontale")

    # Vista scelta per mostrare z orizzontale
    ax.view_init(elev=18, azim=-70)

    plt.tight_layout()
    plt.show()


'''
draw_detector(
    z_planes=z_planes,
    plane_size=100.0,
    tof_margin=50.0,
    x_hits=x_hits,
    y_hits=y_hits,
    n_tracks_show=20
)
'''

'\ndraw_detector(\n    z_planes=z_planes,\n    plane_size=100.0,\n    tof_margin=50.0,\n    x_hits=x_hits,\n    y_hits=y_hits,\n    n_tracks_show=20\n)\n'

In [ ]:
detectorData = env.rollout(1, policy_explore)
print(detectorData['action'])


draw_detector(
    z_planes=detectorData['next', 'observation'][0].numpy(),  # Converti da tensore a numpy
    plane_size=100.0,
    tof_margin=50.0,
    x_hits=None,  # Non abbiamo le hit reali, quindi lasciamo vuoto
    y_hits=None,
    n_tracks_show=0  # Non mostriamo tracce casuali
)

tensor([[2.0000e+00, 1.0000e+00, 1.0000e-02, 1.2184e+01]],
       grad_fn=<StackBackward0>)
